<a href="https://colab.research.google.com/github/vencov/FAV_upsidedown/blob/main/TutDPOAE04/TutDPOAE_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DPOAE thresholds from growth functions: comparison with the audiogram

**Goal:** estimate the DPOAE threshold at several frequencies from the NL component measured with the
scissor paradigm, and compare it with the pure-tone audiogram (PTA) of the same ear.

**Idea:** near threshold, the DPOAE amplitude in *linear* units (µPa) grows approximately linearly with the
stimulus level in dB. A straight line fitted to this part of the growth function and extrapolated to
**0 µPa** crosses the level axis at the estimated DPOAE threshold. Zero pressure exists only on a linear axis
(0 µPa is −∞ dB SPL), so the fit is done in µPa, not in dB SPL.

## 1. Settings and data download
The DPgrams (`.pkl`) and the audiogram (`.txt`) are downloaded from GitHub into the folder `data/`. Set `ear` to the ear that was measured.

In [ ]:
import os
import pickle
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

# ---- settings: edit these ----
# Files are downloaded from the GitHub repository vencov/FAV_upsidedown, folder TutDPOAE04/2026/
# (change 'main' to the branch name if the repository uses a different one)
GITHUB_BASE = "https://raw.githubusercontent.com/vencov/FAV_upsidedown/main/TutDPOAE04/2026/"  # SET PROPER PATH
data_file = "DPgrams_s004SPR.pkl"                 # measured DPgrams (FILL IN YOUR MEASURE DATA, WORKS ONLY FOR ONE DATA SET)
pta_file  = "s004_PTA_20250430_153146.txt"        # pure-tone audiogram
data_dir  = "data/"                               # local folder for the downloaded files

ear = 'right ear'                                 # measured ear: 'right ear' or 'left ear' (selects the PTA lines)
subject_name = data_file.replace("DPgrams_", "")[:4]   # e.g. s004

f2f1 = 1.2                       # f2/f1 ratio
pREF = np.sqrt(2) * 2e-5         # peak reference pressure (Pa)
Pa2uPa = 1e6

label_fontsize, legend_fontsize, text_fontsize = 14, 10, 13

# ---- download the files (skipped if they are already in data_dir) ----
os.makedirs(data_dir, exist_ok=True)
for fname in (data_file, pta_file):
    local = os.path.join(data_dir, fname)
    if os.path.exists(local):
        print("using local copy of", fname, "(delete the data folder to download again)")
        continue
    url = GITHUB_BASE + fname
    try:
        urllib.request.urlretrieve(url, local)
        print("downloaded", fname)
    except Exception as e:
        if os.path.exists(local):
            os.remove(local)
        raise RuntimeError(f"Could not download {url}\n{e}\n"
                           "Check the file name, the branch name in GITHUB_BASE and that the file is pushed to GitHub.") from e

## 2. Load the measured DPgrams and list the level combinations

In [ ]:
with open(os.path.join(data_dir, data_file), "rb") as f:
    DPgr = pickle.load(f)

def num_key(s):
    """Sort level keys numerically (the keys are strings)."""
    try:
        return (0, float(s))
    except ValueError:
        return (1, s)

def level_keys(d):
    """Level keys only (skip keys ending in 'ch'), sorted numerically."""
    return sorted((k for k in d if not str(k).endswith("ch")), key=num_key)

L2_keys = level_keys(DPgr)
n_total = 0
print(f"File: {data_file}")
print(f"Number of L2 levels: {len(L2_keys)}\n")
for L2 in L2_keys:
    L1_keys = level_keys(DPgr[L2])
    n_total += len(L1_keys)
    print(f"L2 = {L2} dB: L1 = {', '.join(L1_keys)} dB")
print(f"\nTotal level combinations: {n_total}")

## 3. Scissor-paradigm level combinations and their DPgrams
Left column: entire DPgram (`DPgr`), right column: NL component (`NLgr`). Dotted lines are noise floors.

In [ ]:
# Scissor paradigm: L1 = a*L2 + b
a, b = 0.4, 39
tol = 0.5          # dB tolerance, the keys are rounded integers

scissor = []       # list of (L2, L1) key strings, ascending L2
for sL2 in level_keys(DPgr):
    for sL1 in level_keys(DPgr[sL2]):
        try:
            l2, l1 = float(sL2), float(sL1)
        except ValueError:
            continue
        if abs(l1 - (a*l2 + b)) <= tol:
            scissor.append((sL2, sL1))
scissor.sort(key=lambda p: float(p[0]))

print("Scissor-paradigm combinations found:")
for sL2, sL1 in scissor:
    print(f"  L2 = {sL2} dB, L1 = {sL1} dB")
L2_vals = np.array([float(sL2) for sL2, sL1 in scissor])

In [ ]:
cycle = 2*np.pi
F2start = 700                                   # Hz, start of phase unwrapping
colors = plt.cm.viridis(np.linspace(0, 0.9, len(scissor)))
xticks = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000]

fig, axs = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
(axA1, axN1), (axA2, axN2) = axs        # rows: amplitude, phase; columns: DPgr (total), NLgr (NL component)

for (sL2, sL1), col in zip(scissor, colors):
    d = DPgr[sL2][sL1]
    fxx = d['fxx']
    f2xx = fxx[:int(len(fxx)//2)+1] / (2 - f2f1)
    idx1 = np.where(f2xx >= F2start)[0][0]
    lab = f'L1={sL1}, L2={sL2} dB'
    for ax_amp, ax_ph, key, keyN in ((axA1, axA2, 'DPgr', 'DPgrN'), (axN1, axN2, 'NLgr', 'NLgrN')):
        ax_amp.plot(f2xx, 20*np.log10(np.abs(d[key])/pREF), color=col, label=lab)
        if keyN in d:
            ax_amp.plot(f2xx, 20*np.log10(np.abs(d[keyN])/pREF), ':', color=col, label='_nolegend_')
        ax_ph.plot(f2xx[idx1:], np.unwrap(np.angle(d[key][idx1:len(f2xx)]))/cycle, color=col, label=lab)

axA1.set_title('DPgram (total)', fontsize=label_fontsize)
axN1.set_title('NL component', fontsize=label_fontsize)
for ax in (axA1, axN1):
    ax.set_ylim([-40, 20]);  ax.set_ylabel('Amplitude (dB SPL)', fontsize=label_fontsize)
for ax in (axA2, axN2):
    ax.set_ylim([-5, 1]);  ax.set_ylabel('Phase (cycles)', fontsize=label_fontsize)
    ax.set_xlabel('Frequency $f_{2}$ (kHz)', fontsize=label_fontsize)
for ax in axs.flat:
    ax.set_xlim([500, 8000]);  ax.set_xticks(xticks);  ax.set_xticklabels([1, 2, 3, 4, 5, 6, 7, 8])
axA1.legend(fontsize=legend_fontsize, ncol=2)
axA2.text(0.3, 0.2, f'{subject_name} ({ear})', transform=axA2.transAxes,
          fontsize=text_fontsize, verticalalignment='top', horizontalalignment='right')
fig.tight_layout()
plt.show()

## 4. Growth functions of the NL component (µPa)
Filled markers: NL at least `snr_crit` dB above the noise floor (dotted). Open markers: below this criterion.

In [ ]:
def get_growth(f2_target, pairs):
    """NL component and its noise floor (µPa) at the bin nearest to f2_target, along the scissor path."""
    nl, noise = [], []
    for sL2, sL1 in pairs:
        d = DPgr[sL2][sL1]
        f2xx = d['fxx'][:int(len(d['fxx'])//2)+1] / (2 - f2f1)
        idx = np.argmin(np.abs(f2xx - f2_target))
        nl.append(np.abs(d['NLgr'][idx]) * Pa2uPa)
        noise.append(np.abs(d['NLgrN'][idx]) * Pa2uPa)
    return np.array(nl), np.array(noise)

f2_targets = [1000, 1500, 2000, 3000, 4000, 6000]
snr_crit = 6       # dB; open markers = NL less than this above the noise floor

fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(f2_targets)))
for f2t, col in zip(f2_targets, colors):
    nl, noise = get_growth(f2t, scissor)
    ax.plot(L2_vals, nl, '-', color=col, label=f'$f_2$ = {f2t/1000:g} kHz')
    ax.plot(L2_vals, noise, ':', color=col, label='_nolegend_')
    good = 20*np.log10(nl / noise) >= snr_crit
    ax.plot(L2_vals[good], nl[good], 'o', color=col, label='_nolegend_')
    ax.plot(L2_vals[~good], nl[~good], 'o', mfc='none', color=col, label='_nolegend_')
ax.set_yscale('log')
ax.set_xlabel('$L_2$ (dB SPL)', fontsize=label_fontsize)
ax.set_ylabel('NL component amplitude (µPa)', fontsize=label_fontsize)
ax.set_title(f'DPOAE growth functions, NL component, {subject_name} ({ear})', fontsize=label_fontsize)
ax.legend(fontsize=legend_fontsize, loc='upper left')
ax.grid(alpha=0.3, which='both')
fig.tight_layout()
plt.show()

## 5. Automatic threshold estimate HOMEWORK
Fit a line to the lowest-level points above the noise floor. Add points one by one and stop when the slope of the cumulative fit drops by more than `max_drop` relative to the slope of the first `k_min` points, which indicates compression.

In [ ]:

def fit_auto(L2, nl_uPa, noise_uPa, snr_crit=6, k_min=3, max_drop=0.2):
    """Estimate the DPOAE threshold from one growth function.

    L2        : array of L2 levels (dB SPL)
    nl_uPa    : NL component amplitude (µPa, LINEAR units!)
    noise_uPa : noise floor at the same points (µPa)

    The line is fitted to the lowest-level points that are above the noise floor.
    Points are added one by one until the slope of the cumulative fit drops by more
    than `max_drop` relative to the slope of the first `k_min` points.
    Returns a dict with the fit results, or None if no fit is possible.
    """

    # Keep points at least snr_crit dB above the noise floor, sorted by L2
    ok = 20*np.log10(nl_uPa / noise_uPa) >= snr_crit
    order = np.argsort(L2[ok])
    x, y = L2[ok][order], nl_uPa[ok][order]

    if x.size < k_min:
        return None

    # Reference slope from the lowest-level points
    slope0, _ = np.polyfit(x[:k_min], y[:k_min], 1)

    if slope0 <= 0:
        return None

    n_used = k_min

    for k in range(k_min + 1, len(x) + 1):
        slope, _ = np.polyfit(x[:k], y[:k], 1)

        # TODO 1: Detect compression
        compressed =

        if compressed:
            break  # The k-th point is already in the saturated part

        n_used = k

    # TODO 2: Fit the final line using accepted points only
    slope, intercept =

    # TODO 3: Calculate the threshold (dB SPL)
    threshold =

    if slope <= 0:
        return None
    return dict(x=x, y=y, n_used=n_used, slope=slope, intercept=intercept, threshold=threshold)

### Self-test

In [ ]:
# Self-test on synthetic data: linear growth (2 µPa/dB, zero at L2 = 20 dB), saturating above 50 dB
L2_test = np.arange(25, 75, 5.0)
y_test = np.where(L2_test <= 50, 2.0*(L2_test - 20), 60 + 0.1*(L2_test - 50))
noise_test = np.full_like(y_test, 1.0)

r = fit_auto(L2_test, y_test, noise_test)
print(f"threshold = {r['threshold']:.1f} dB (true value: 20 dB), points used: {r['n_used']}, slope = {r['slope']:.2f} µPa/dB")
assert abs(r['threshold'] - 20) < 3, "Threshold is far from the true value, check your fit."
print("Self-test passed.")

## 6. Growth functions with fits and estimated thresholds
Orange: points used in the fit. Grey: points above the noise floor excluded by the slope criterion. Open blue: points too close to the noise floor. Red square: estimated threshold.

In [ ]:
snr_crit, k_min, max_drop = 6, 3, 0.2
results = []                      # (f2, threshold, n_used, slope)

fig, axs = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
for ax, f2t in zip(axs.flat, f2_targets):
    nl, noise = get_growth(f2t, scissor)
    res = fit_auto(L2_vals, nl, noise, snr_crit, k_min, max_drop)

    snr_ok = 20*np.log10(nl / noise) >= snr_crit
    ax.plot(L2_vals, noise, ':', color='gray', label='noise floor')
    ax.plot(L2_vals[~snr_ok], nl[~snr_ok], 'o', mfc='none', color='C0', label='below SNR crit.')
    ax.axhline(0, color='k', lw=0.5)

    if res is None:
        ax.set_title(f'$f_2$ = {f2t/1000:g} kHz: no fit', fontsize=label_fontsize)
        results.append((f2t, np.nan, 0, np.nan))
        continue

    xs = res['x']
    used = np.arange(len(xs)) < res['n_used']
    ax.plot(xs[~used], res['y'][~used], 'o', color='gray', label='excluded (compression)')
    ax.plot(xs[used], res['y'][used], 'o', color='C1', label='used in fit')
    xl = np.array([min(res['threshold'], xs[0]) - 3, xs[-1] + 2])
    ax.plot(xl, res['slope']*xl + res['intercept'], '-', color='C1')
    ax.plot(res['threshold'], 0, 's', color='C3', ms=8, label=f"threshold = {res['threshold']:.1f} dB")
    ax.set_title(f'$f_2$ = {f2t/1000:g} kHz', fontsize=label_fontsize)
    ax.legend(fontsize=8, loc='upper left')
    results.append((f2t, res['threshold'], res['n_used'], res['slope']))

for ax in axs[1]:
    ax.set_xlabel('$L_2$ (dB SPL)', fontsize=label_fontsize)
for ax in axs[:, 0]:
    ax.set_ylabel('NL amplitude (µPa)', fontsize=label_fontsize)
fig.suptitle(f'DPOAE growth functions and fits, {subject_name} ({ear})', fontsize=label_fontsize)
fig.tight_layout()
plt.show()

print("f2 [Hz]  threshold [dB SPL]  n_used  slope [µPa/dB]")
for f2t, thr, n, s in results:
    print(f"{f2t:>7}  {thr:>17.1f}  {n:>6}  {s:>13.2f}")

## 7. Comparison with the audiogram
The PTA values are given in dB re absolute threshold. Adding the RETSPL of the headphones converts them to dB SPL, which is the scale of the DPOAE threshold ($L_2$).

In [ ]:
# RETSPL of the headphones (dB SPL)
ATfreq = np.array([125, 250, 500, 750, 1000, 1500, 2000, 3000, 4000, 5000,
                   6000, 8000, 9000, 10000, 11200, 12500, 14000, 16000])
AThresh = np.array([26.2, 20.1, 8.6, 5.1, 2.7, 3.2, 0.5, -1.6, 0.1, 11.3,
                    20.9, 23.1, 27.1, 18.5, 22.9, 27.0, 32.8, 47.7])
retspl = dict(zip(ATfreq, AThresh))

def load_pta(path, ear_name):
    """Read the PTA text file; return {freq_Hz: level dB re abs. threshold} for one ear."""
    pta = {}
    with open(path) as f:
        for line in f:
            parts = [p.strip() for p in line.split(',')]
            if len(parts) == 3 and parts[0].lower() == ear_name.lower():
                pta[int(float(parts[1]))] = float(parts[2])
    return pta

ear_name = 'Right Ear' if ear.startswith('right') else 'Left Ear'
pta_hl = load_pta(os.path.join(data_dir, pta_file), ear_name)
if not pta_hl:
    raise ValueError(f"No '{ear_name}' lines found in {pta_file}")

# dB SPL = level re abs. threshold + RETSPL
pta_spl = {}
for fr, lev in pta_hl.items():
    if fr in retspl:
        pta_spl[fr] = lev + retspl[fr]
    else:
        print(f"No RETSPL for {fr} Hz, skipped")

dp_thr = {int(f2t): thr for f2t, thr, n, s in results if not np.isnan(thr)}
common = sorted(set(pta_spl) & set(dp_thr))

print("f [Hz]  PTA [dB SPL]  DPOAE thr [dB SPL]")
for fr in sorted(set(pta_spl) | set(dp_thr)):
    p = f"{pta_spl[fr]:>12.1f}" if fr in pta_spl else f"{'-':>12}"
    d = f"{dp_thr[fr]:>18.1f}" if fr in dp_thr else f"{'-':>18}"
    print(f"{fr:>6}  {p}  {d}")

fig, (axF, axS) = plt.subplots(1, 2, figsize=(13, 5))

fp = sorted(pta_spl)
axF.plot(fp, [pta_spl[f] for f in fp], 'o-', color='C0', label='PTA (dB SPL)')
fd = sorted(dp_thr)
axF.plot(fd, [dp_thr[f] for f in fd], 's-', color='C1', label='DPOAE threshold ($L_2$, dB SPL)')
axF.set_xscale('log')
axF.set_xticks([500, 1000, 2000, 4000, 8000]);  axF.set_xticklabels([0.5, 1, 2, 4, 8])
axF.set_xlabel('Frequency (kHz)', fontsize=label_fontsize)
axF.set_ylabel('Threshold (dB SPL)', fontsize=label_fontsize)
axF.legend(fontsize=legend_fontsize);  axF.grid(alpha=0.3)

if common:
    x = np.array([pta_spl[f] for f in common])
    y = np.array([dp_thr[f] for f in common])
    axS.plot(x, y, 'o', color='C1')
    for f, xi, yi in zip(common, x, y):
        axS.annotate(f'{f/1000:g} kHz', (xi, yi), textcoords='offset points', xytext=(5, 5), fontsize=legend_fontsize)
    lim = [min(x.min(), y.min()) - 5, max(x.max(), y.max()) + 5]
    axS.plot(lim, lim, 'k--', lw=0.8, label='identity')
    axS.set_xlim(lim);  axS.set_ylim(lim);  axS.set_aspect('equal')
    axS.legend(fontsize=legend_fontsize)
axS.set_xlabel('PTA (dB SPL)', fontsize=label_fontsize)
axS.set_ylabel('DPOAE threshold (dB SPL)', fontsize=label_fontsize)
axS.grid(alpha=0.3)

fig.suptitle(f'{subject_name} ({ear})', fontsize=label_fontsize)
fig.tight_layout()
plt.show()

## Questions

1. Why is the line fitted to the amplitude in µPa and not in dB SPL?
2. Why must the points be sorted by $L_2$ before the loop?
3. Change `max_drop` to 0.1 and 0.3. How do the thresholds and `n_used` change? At which frequencies?
4. Change `snr_crit` to 3 and 10 dB. How does this affect the number of points and the threshold?
5. At which frequencies does the slope criterion stop first? Does this agree with where you expect stronger compression?
6. Is any estimated threshold below the lowest measured $L_2$? How far can an extrapolated value be trusted?
7. Compare the DPOAE thresholds with the PTA (in dB SPL). Do they follow the same trend across frequency? Why are DPOAE thresholds usually higher than the PTA?
8. **Extension:** estimate the uncertainty of the threshold with a bootstrap (resample the points with replacement, refit, repeat 1000 times, report the 2.5 and 97.5 percentiles). Why is the interval asymmetric, and why can it be too narrow with only 4-6 points?